# 전체 데이터 EDA (eda_full)

`data_fix/tables/` (10/22 이전: `data/all_data_zip/population/population/`) 아래 전체 덤프(API 1,811표 + 대용량 ZIP 553표)를 표 단위로 훑은 결과를 읽는 노트북입니다.

**순서**
1. 무거운 작업(프로파일링)은 노트북이 아니라 **터미널에서** `scripts/eda_full.py` 로 돌립니다. 몇 시간 걸려도 되고, 중간에 멈추면 같은 명령을 다시 치면 이어서 합니다.
2. 끝나면 이 노트북을 위에서부터 실행합니다. 이 노트북은 `data/eda_full/summary/*.csv` 와 `data/eda_full/profiles/*.json` 만 읽으므로 몇 초~1분이면 끝납니다.

**터미널 실행 (Anaconda Prompt)**
```
cd "C:\Users\ckawo\OneDrive\바탕 화면\KOSIS-ai-nlp-summary"
conda activate kosis
python scripts/eda_full.py --limit 20     # 먼저 20표만 시험 (1~2분)
python scripts/eda_full.py                # 전체 실행 (몇 시간, 이어하기 지원)
```
- 실행 중 Windows 절전 모드가 켜지면 멈추므로, 설정 > 전원에서 절전을 '안 함'으로 바꿔 두세요.
- 어떤 표에서 오류가 나도 그 표만 `errors.jsonl` 에 적고 다음 표로 넘어갑니다.
- 오류 난 표만 다시: `python scripts/eda_full.py --retry-errors`
- 요약 CSV만 다시 만들기: `python scripts/eda_full.py --summarize-only`

## 0. 설정과 진행 상황 확인

**무엇을**: 결과 폴더 경로를 정하고, 프로파일이 몇 개 만들어졌는지, 로그 마지막 줄이 무엇인지 봅니다.

**어떻게**: `data/eda_full/profiles/` 의 json 개수를 세고 `run_log.txt` 끝 5줄을 출력합니다. 실행 도중에 돌려도 안전합니다(읽기만 함).

**결과 읽는 법**: 프로파일 수가 2,364에 가까우면 거의 끝난 것입니다. 로그 마지막 줄에 `summary written` 이 있으면 요약 CSV까지 만들어진 상태입니다. `errors.jsonl` 줄 수는 처리에 실패한 표 수입니다.

In [ ]:
import ast
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "data" / "eda_full"
SUM = OUT / "summary"
PROF = OUT / "profiles"

n_prof = len(list(PROF.glob("*.json"))) if PROF.exists() else 0
print("repo:", REPO)
print("profiles:", n_prof, "/ 2364")
log = OUT / "run_log.txt"
if log.exists():
    print("--- run_log.txt (마지막 5줄)")
    print("\n".join(log.read_text(encoding="utf-8").splitlines()[-5:]))
err = OUT / "errors.jsonl"
print("errors.jsonl:", sum(1 for _ in err.open(encoding="utf-8")) if err.exists() else 0, "줄")

## 1. 요약 CSV 불러오기

**무엇을**: 스크립트가 만든 요약 표 6개를 읽습니다.
- `tables.csv`: 표 1개 = 1행 (관측치 수, 차원 수, 시점, 단위, 기호, 결측, 합계 검증 등 핵심 지표)
- `dims.csv`: 표×차원 1행 (코드 수, 코드 길이 분포, '계' 같은 합계 라벨)
- `units.csv`, `symbols.csv`, `periods.csv`, `sum_checks.csv`: 단위·특수기호·시점·합계검증 상세

**어떻게**: `utf-8-sig` 로 읽고, 문자열로 저장된 dict/list 칸(예: `code_lengths`)을 파이썬 객체로 되돌립니다.

**결과 읽는 법**: 각 표의 행 수가 출력됩니다. `tables` 행 수는 처리된 표 수와 같아야 합니다.

In [ ]:
def read(name):
    p = SUM / f"{name}.csv"
    if not p.exists() or p.stat().st_size == 0:
        print(f"(없음) {p}")
        return pd.DataFrame()
    return pd.read_csv(p, encoding="utf-8-sig", low_memory=False)

def parse_obj(x):
    if isinstance(x, str) and x[:1] in "[{(":
        try:
            return ast.literal_eval(x)
        except Exception:
            return x
    return x

tables = read("tables")
dims = read("dims")
units = read("units")
symbols = read("symbols")
periods = read("periods")
sums = read("sum_checks")
for df, cols in ((dims, ["code_lengths", "total_like", "paren_units"]), (periods, ["shapes", "gap_examples"]),
                 (sums, ["total_labels", "ratio_sum_over_total_p10_p50_p90"])):
    for c in cols:
        if c in df:
            df[c] = df[c].map(parse_obj)
ok = tables[tables["status"] == "ok"].copy() if len(tables) else tables
for name, df in (("tables", tables), ("dims", dims), ("units", units), ("symbols", symbols), ("periods", periods), ("sum_checks", sums)):
    print(f"{name:10s} {len(df):>7,} 행")

## 2. 수집·처리 현황 (데이터가 실제로 있는 표는 몇 개인가)

**무엇을**: 표마다 (a) 스크립트가 처리에 성공했는지(`status`), (b) 실제 값이 들어 있는지(`data_state`), (c) API 수집 상태(`api_status`)를 셉니다.

**어떻게**: `data_state` 는 `full`(값 있음), `partial`(수집이 중간에 끊겨 일부만 있음: API 상태 running/incomplete), `no_data`(값 0개: 수집 실패 표) 세 가지입니다.

**결과 읽는 법**
- `status=error` 는 스크립트가 읽다가 실패한 표 → 아래 오류 목록을 보고 원인을 판단합니다(파일 손상, 형식 예외 등).
- `no_data` 는 덤프 자체에 값이 없는 표(안내된 결손 약 3%)입니다. 챗봇은 이 표에 대해 '데이터 없음'으로 답하거나 API로 따로 받아야 합니다.
- `partial` 은 일부 시점만 있는 표라, 시점 커버리지(6절)를 같이 봐야 합니다.

In [ ]:
print(tables.groupby(["source", "status"]).size().unstack(fill_value=0), "\n")
if "data_state" in ok:
    print(ok.groupby(["source", "data_state"]).size().unstack(fill_value=0), "\n")
if "api_status" in ok:
    print(ok[ok["source"] == "api"]["api_status"].value_counts(dropna=False), "\n")
bad = tables[tables["status"] != "ok"]
if len(bad):
    print("처리 오류 표:")
    display(bad[["key", "source", "survey", "title", "error"]])
if "data_state" in ok:
    display(ok[ok["data_state"] != "full"][["key", "source", "survey", "title", "data_state", "api_status", "api_chunk_source", "observations"]])

## 3. 조사별 개요

**무엇을**: 통계조사(survey)마다 표 수, 값(관측치) 수, 원본 크기, 프롬프트 토큰 추정치를 합산합니다.

**어떻게**: `ok` 표를 `survey` 로 묶어 합계·중앙값을 냅니다. `approx_prompt_tokens` 는 '표 전체를 라벨 포함 텍스트로 LLM에 넣으면 대략 몇 토큰인가'를 글자 수/2 로 어림한 값입니다(한국어 기준 거친 추정). '값 하나마다 차원·항목 라벨을 모두 적은 한 줄' 형식 기준이라 상한에 가까운 값이고, 라벨을 머리글로 묶은 표 형식이면 몇 배 줄어듭니다.

**결과 읽는 법**: 관측치·토큰이 큰 조사일수록 표를 통째로 넣을 수 없어 '필요한 부분만 잘라 넣기'(필터링/RAG)가 필수입니다. 표 수가 많은 조사는 표 선택(검색) 난이도가 높습니다.

In [ ]:
g = ok.groupby("survey", dropna=False).agg(
    tables=("key", "count"),
    observations=("observations", "sum"),
    median_obs=("observations", "median"),
    mbytes=("bytes", lambda s: round(s.sum() / 1e6, 1)),
    tokens_sum=("approx_prompt_tokens", "sum"),
    tokens_median=("approx_prompt_tokens", "median"),
).sort_values("observations", ascending=False)
print("전체 관측치:", f"{int(ok['observations'].sum()):,}", "/ 전체 토큰 추정:", f"{int(ok['approx_prompt_tokens'].sum()):,}")
display(g)

## 4. 표 크기 분포 — LLM 컨텍스트에 통째로 들어가는 표는 몇 개인가

**무엇을**: 표별 관측치 수와 토큰 추정치를 구간으로 나눠 셉니다.

**어떻게**: 토큰 구간은 8K / 32K / 128K / 1M 기준입니다(일반적인 모델 컨텍스트 크기). 상위 15개 큰 표도 보여 줍니다.

**결과 읽는 법**: `≤8K` 표는 프롬프트에 그대로 넣어도 되는 표, `>128K` 표는 반드시 차원·시점 필터링이 필요한 표입니다. 이 비율이 T1-05(전처리)와 2차(RAG) 설계의 출발점입니다.

In [ ]:
bins = [-1, 8_000, 32_000, 128_000, 1_000_000, float("inf")]
labels = ["≤8K", "8K~32K", "32K~128K", "128K~1M", ">1M"]
ok["token_band"] = pd.cut(ok["approx_prompt_tokens"], bins=bins, labels=labels)
print(ok.groupby(["source", "token_band"], observed=False).size().unstack(fill_value=0), "\n")
print(ok["observations"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0), "\n")
display(ok.nlargest(15, "observations")[["key", "survey", "title", "observations", "approx_prompt_tokens", "n_dims", "max_codes_in_dim", "n_periods_max"]])

## 5. 차원 구조 — 몇 개 축으로 쪼개져 있고, 계층은 어떻게 표현되는가

**무엇을**
- 표당 차원 수(`n_dims`)와 항목 수(`n_items`) 분포
- 자주 나오는 차원 이름 (행정구역, 성별, 연령 … ) — 질문 해석에서 슬롯으로 쓸 후보
- 한 차원 안에 코드 길이가 여러 개인 경우(`n_code_lengths ≥ 2`) = 코드 앞자리로 상하위 계층을 표현하는 차원 (예: 시도 2자리 + 시군구 5자리)
- 같은 라벨을 서로 다른 코드가 쓰는 경우(`labels_reused`) = 예: 여러 시도의 '동부/읍부/면부', 일반구 → 라벨만으로 구분 불가
- '계/합계/전국' 같은 합계 라벨이 없는 차원

**어떻게**: `dims.csv` 를 집계합니다.

**결과 읽는 법**: 계층 차원과 라벨 재사용 차원이 많을수록 전처리에서 '부모 라벨을 붙인 전체 이름'(예: 서울특별시 > 종로구)을 만들어야 합니다. 합계 라벨이 없는 차원은 '전체'를 물었을 때 직접 합산해야 할 수 있습니다.

In [ ]:
print("표당 차원 수:\n", ok["n_dims"].value_counts().sort_index(), "\n")
print("표당 항목 수:\n", ok["n_items"].describe(percentiles=[0.5, 0.9, 0.99]).round(1), "\n")
top = dims.groupby("dim").agg(tables=("key", "nunique"), median_codes=("codes", "median"), max_codes=("codes", "max")) \
          .sort_values("tables", ascending=False).head(30)
display(top)

In [ ]:
dims["hierarchical"] = dims["n_code_lengths"] >= 2
dims["has_total_label"] = dims["total_like"].map(lambda x: bool(x) if isinstance(x, (list, tuple)) else False)
print("코드 길이가 2종 이상인 차원(계층 추정):", int(dims["hierarchical"].sum()), "/", len(dims))
print("라벨 재사용이 있는 차원:", int((dims["labels_reused"] > 0).sum()))
print("합계 라벨이 없는 차원:", int((~dims["has_total_label"]).sum()), "\n")
print("계층 차원의 코드 길이 조합 상위 15:")
print(dims[dims["hierarchical"]]["code_lengths"].map(lambda d: tuple(sorted(map(int, d))) if isinstance(d, dict) else d)
      .value_counts().head(15), "\n")
display(dims[dims["labels_reused"] > 0].sort_values("labels_reused", ascending=False)
        [["key", "title", "dim", "codes", "labels_reused", "code_lengths"]].head(20))

## 6. 시점 — 주기, 기간, 최신성, 빠진 시점

**무엇을**
- 주기별 표 수 (Y=연, M=월, Q=분기, H=반기, F=다년(5년 등), 기타)
- 시작·마지막 시점의 분포: 최신 자료가 언제까지 있는가
- 주기 안에서 중간에 빠진 시점(`gaps`): 예) 연간 표인데 2019가 비어 있음
- 시점 문자열 모양(`shapes`): 예) `9999`, `999999`, `99990Q` → 파서가 모두 처리해야 할 형식

**어떻게**: `periods.csv`(표×주기 1행)를 집계합니다. 빈 시점은 연/월/분기/반기 주기에서만 계산합니다.

**결과 읽는 법**: gap 이 있는 표는 '전년 대비'를 계산할 때 바로 앞 시점이 없을 수 있으니, 답변에서 비교 기준 시점을 명시해야 합니다. 마지막 시점이 오래된 표는 '최신' 질문에 주의가 필요합니다.

In [ ]:
print("주기별 표 수:\n", periods.groupby("cycle")["key"].nunique().sort_values(ascending=False), "\n")
print("주기 조합(표 기준):\n", ok["cycles"].value_counts().head(10), "\n")
periods["last_year"] = periods["last"].astype(str).str[:4]
print("마지막 시점 연도 분포(상위):\n", periods["last_year"].value_counts().sort_index(ascending=False).head(15), "\n")
shp = periods["shapes"].explode().value_counts()
print("시점 문자열 모양:\n", shp, "\n")
gp = periods[periods["gaps"].fillna(0) > 0].sort_values("gaps", ascending=False)
print("빈 시점이 있는 표×주기:", len(gp))
display(gp[["key", "title", "cycle", "count", "first", "last", "gaps", "gap_examples"]].head(25))

## 7. 단위 — 값에 단위가 붙어 있는가

**무엇을**
- `unit_field_missing_obs`: API 응답에 `UNIT_NM` 필드 자체가 없는 관측치 수 (기존 B0 로더가 실패하는 원인)
- `unit_blank_obs`: 필드는 있지만 빈 문자열
- `items_with_several_units`: 한 항목에 단위가 2개 이상 섞인 경우
- 메타데이터 단위(`unit_meta`)와 실제 관측 단위의 비교
- 가장 흔한 단위 목록 (정규화 사전의 후보)

**어떻게**: `tables.csv` 지표와 `units.csv`(표×단위 1행)를 봅니다. ZIP(대용량) 표는 항목 열 이름 괄호 안 단위를 단위로 씁니다.

**결과 읽는 법**: 단위가 비거나 없는 표가 많으면 전처리(T1-05 R5)에서 '항목명 괄호 → 메타 단위 → 없음 표시' 순서의 보정 규칙이 필수입니다. 한 표에 여러 단위가 섞이면 항목별로 단위를 따로 붙여야 합니다.

In [ ]:
no_unit = ok[(ok["n_units"] == 0) & (ok["observations"] > 0)]
print("값은 있는데 단위가 하나도 없는 표:", len(no_unit), "/", int((ok["observations"] > 0).sum()))
print("UNIT_NM 필드가 없는 관측치가 있는 표:", int((ok["unit_field_missing_obs"] > 0).sum()))
print("단위가 빈 문자열인 관측치가 있는 표:", int((ok["unit_blank_obs"] > 0).sum()))
print("한 항목에 단위가 여러 개인 표:", int((ok["items_with_several_units"] > 0).sum()))
print("표 안 단위 종류 수:\n", ok["n_units"].value_counts().sort_index(), "\n")
u = units[~units["unit"].isin(["", "<필드 없음>"])].groupby("unit").agg(tables=("key", "nunique"), observations=("observations", "sum"))
display(u.sort_values("tables", ascending=False).head(40))

In [ ]:
# 메타 단위와 실제 단위가 다른 표 (메타 단위가 있는 표만)
obs_units = units[~units["unit"].isin(["", "<필드 없음>"])].groupby("key")["unit"].apply(lambda s: sorted(set(s)))
cmp = ok[["key", "title", "unit_meta"]].copy()
cmp["observed_units"] = cmp["key"].map(obs_units)
cmp["unit_meta"] = cmp["unit_meta"].fillna("")
has_meta = cmp[cmp["unit_meta"] != ""]
diff = has_meta[has_meta.apply(lambda r: isinstance(r["observed_units"], list) and r["unit_meta"] not in r["observed_units"], axis=1)]
print("메타 단위가 있는 표:", len(has_meta), "/ 관측 단위와 다른 표:", len(diff))
display(diff.head(25))

## 8. 값과 특수기호 — 숫자가 아닌 값은 얼마나, 어떤 모양인가

**무엇을**
- `numeric_share`: 숫자로 읽히는 값의 비율
- 특수기호: `-`(해당 없음/0), `x`(비밀보호), `…`(미상), 빈 값, `p)` 등 잠정치 표시, 그 밖의 문자열
- 음수·0 개수, 소수점 자릿수 (반올림 규칙에 필요)

**어떻게**: `tables.csv` 의 기호 열과 `symbols.csv`(표×기호 1행)를 집계합니다. 기타 문자열 예시는 표별 프로파일 json 의 `values.other_text_examples` 에 있습니다.

**결과 읽는 법**: 기호가 있는 표는 LLM이 '-'를 0으로, 'x'를 값으로 오해하지 않도록 전처리에서 의미 라벨(예: '비공개')로 바꿔 넣어야 합니다. 소수 자릿수가 큰 표는 답변에서 원 자릿수를 유지하는 규칙이 필요합니다.

In [ ]:
print("숫자 비율 분포:\n", ok["numeric_share"].describe(percentiles=[0.01, 0.05, 0.5]).round(4), "\n")
cols = ["provisional", "blank", "dash", "x_secret", "ellipsis", "other_text"]
print("기호가 1개 이상 있는 표 수:")
print((ok[cols] > 0).sum(), "\n")
print("기호별 전체 관측치:")
print(ok[cols].sum(), "\n")
print("기호 원문 상위:")
display(symbols.groupby("symbol").agg(tables=("key", "nunique"), observations=("observations", "sum"))
        .sort_values("tables", ascending=False).head(30))
print("음수가 있는 표:", int((ok["negative"] > 0).sum()), "/ 최대 소수 자릿수 분포:\n", ok["max_decimals"].value_counts().sort_index())

In [ ]:
# 기타 문자열(숫자도 정해진 기호도 아닌 값) 예시를 프로파일에서 모아 보기
rows = []
for k in ok[ok["other_text"] > 0]["key"]:
    try:
        p = json.loads((PROF / f"{k}.json").read_text(encoding="utf-8"))
    except Exception:
        continue
    for txt, n in (p.get("values", {}).get("other_text_examples") or {}).items():
        rows.append({"key": k, "text": txt, "n": n})
other = pd.DataFrame(rows)
if len(other):
    display(other.groupby("text").agg(tables=("key", "nunique"), n=("n", "sum")).sort_values("tables", ascending=False).head(30))
else:
    print("기타 문자열 없음")

## 9. 격자 완전성 — 차원×항목×시점 조합 중 비어 있는 칸

**무엇을**: 표의 모든 코드·항목·시점을 곱한 '완전한 격자' 대비 실제로 값이 있는 칸의 비율입니다.

**어떻게**: 주기별로 `존재하는 관측치 / (각 차원 코드 수 × 항목 수 × 시점 수)` 를 계산했고, 그중 가장 많이 빈 주기의 비율을 `grid_missing_share_max` 로 저장했습니다. (메모리 한도 `--max-store` 를 넘는 큰 표는 계산하지 않아 빈 칸입니다.)

**결과 읽는 법**: 빈 비율이 높은 표는 '구조적 결측'(예: 어떤 연령대는 특정 연도부터만 집계, 행정구역 개편)일 가능성이 큽니다. 이런 표에서 값이 없을 때 챗봇은 '0'이 아니라 '해당 조합은 집계되지 않음'이라고 답해야 합니다.

In [ ]:
gm = ok["grid_missing_share_max"].dropna()
print("계산된 표:", len(gm), "/ 계산 안 된 표(너무 큼):", int(ok["grid_missing_share_max"].isna().sum()))
print(pd.cut(gm, [-0.001, 0, 0.05, 0.2, 0.5, 0.9, 1.0], labels=["0", "≤5%", "5~20%", "20~50%", "50~90%", ">90%"]).value_counts().sort_index(), "\n")
display(ok.sort_values("grid_missing_share_max", ascending=False)
        [["key", "title", "observations", "n_dims", "cycles", "grid_missing_share_max"]].head(20))

## 10. 합계 검증 — '계'가 하위 항목의 합과 같은가

**무엇을**: 각 차원(그리고 항목 축)에서 '계/합계/전국' 같은 합계 코드 값이, 가장 짧은 코드 길이의 형제 코드들(예: 시도 전체, 남+여) 합과 같은지 확인합니다.

**어떻게**: 나머지 차원·시점이 같은 묶음마다 `합계값` 과 `형제합` 을 비교해서 정확히 같은 비율(`exact`), 0.1% 이내 비율, `형제합/합계` 비율의 10·50·90 백분위를 냈습니다.

**결과 읽는 법**
- exact 가 거의 100%: 그 차원은 안전하게 '합계 = 하위 합'으로 다룰 수 있음.
- 비율이 1보다 크게 나옴(예: 1.3): 하위에 중복 집계 항목(일반구, '동부/읍부/면부' 등)이 섞여 있어 단순 합산하면 과대계산 → 챗봇이 직접 합산하면 안 되는 차원.
- 비율이 1보다 작음: 하위에 '미상/기타'가 빠져 있거나 일부 항목만 공개.
- 평균·비율·밀도 단위(%, 세, 명/㎢ 등) 표는 원래 합이 맞지 않으므로 `additive_unit=False` 로 표시하고 '맞지 않는 목록'에서 뺍니다.

In [ ]:
if len(sums):
    sums["exact_share"] = sums["exact"] / sums["checked"]
    sums["close_share"] = sums["within_0.1pct"] / sums["checked"]
    sums["ratio_p50"] = sums["ratio_sum_over_total_p10_p50_p90"].map(lambda x: x[1] if isinstance(x, (list, tuple)) else None)
    print("검사된 (표, 축):", len(sums), "/ 표:", sums["key"].nunique())
    print(pd.cut(sums["close_share"], [-0.001, 0.5, 0.9, 0.99, 1.0], labels=["≤50%", "50~90%", "90~99%", ">99%"]).value_counts().sort_index(), "\n")
    # 비율·평균·밀도처럼 원래 더할 수 없는 단위는 따로 본다 (메타 단위 또는 관측 단위 기준)
    NON_ADDITIVE = r"%|‰|퍼센트|/|율|비$|배|지수|평균|^세$"
    obs_u = units[~units["unit"].isin(["", "<필드 없음>"])].groupby("key")["unit"].apply(lambda s: " ".join(sorted(set(s))))
    unit_of = ok.set_index("key")["unit_meta"].fillna("")
    sums["unit_meta"] = sums["key"].map(unit_of)
    sums["units_seen"] = sums["key"].map(obs_u).fillna("")
    sums["additive_unit"] = ~(sums["unit_meta"].fillna("") + " " + sums["units_seen"]).str.contains(NON_ADDITIVE, regex=True)
    print("더할 수 있는 단위의 (표, 축):", int(sums["additive_unit"].sum()), "/ 비율·평균 등:", int((~sums["additive_unit"]).sum()))
    bad_sum = sums[(sums["close_share"] < 0.9) & sums["additive_unit"]].sort_values("close_share")
    print("더할 수 있는 단위인데 합계가 맞지 않는 (표, 축):", len(bad_sum))
    display(bad_sum[["key", "title", "axis", "unit_meta", "units_seen", "total_labels", "siblings", "sibling_code_len", "checked", "close_share", "ratio_sum_over_total_p10_p50_p90"]].head(30))
    print("축 이름별 합계 일치율(중앙값):")
    display(sums[sums["additive_unit"]].groupby("axis").agg(n=("key", "count"), close_share_median=("close_share", "median"), ratio_p50_median=("ratio_p50", "median"))
            .sort_values("n", ascending=False).head(20))

## 11. 키 중복과 메타데이터 비교

**무엇을**
- `duplicate_keys`: 같은 (차원 코드들, 항목, 시점) 조합이 두 번 이상 나온 횟수. 0이 아니면 같은 칸에 값이 두 개 → 수집 중복이거나 키 정의가 부족한 것.
- 공식 메타(`table_metadata/*.json`)의 분류 개수와 실제 데이터의 차원 수 비교.

**어떻게**: `tables.csv` 의 `duplicate_keys`, `meta_classes`, `n_dims` 를 봅니다.

**결과 읽는 법**: 중복이 있는 표는 로더에서 중복 제거 규칙(최신 수집본 우선 등)이 필요합니다. 메타 분류 수와 실제 차원 수가 다르면 메타만 보고 질문을 해석하면 틀릴 수 있습니다.

In [ ]:
dup = ok[ok["duplicate_keys"].fillna(0) > 0]
print("키 중복이 있는 표:", len(dup), "/ 메모리 한도로 중복 검사 못한 표:", int((ok["fit_in_memory"] == False).sum()))
display(dup.sort_values("duplicate_keys", ascending=False)[["key", "source", "title", "observations", "duplicate_keys", "api_chunk_source"]].head(20))
has = ok[(ok["meta_classes"] > 0) & (ok["observations"] > 0)]
mm = has[has["meta_classes"] != has["n_dims"]]
print("메타 분류 수 ≠ 실제 차원 수:", len(mm), "/", len(has))
display(mm[["key", "source", "title", "meta_classes", "n_dims", "dim_names"]].head(20))

## 12. 소스별 차이 (API vs 대용량 ZIP)

**무엇을**: 같은 지표를 API 표와 ZIP 표로 나눠 비교합니다.

**어떻게**: `source` 로 묶어 단위 결손, 기호, 차원 수, 크기의 중앙값·비율을 냅니다.

**결과 읽는 법**: ZIP 표는 넓은 형식(열 = 항목×마지막 분류)이라 '항목' 개수가 크게 나오고, 단위는 열 이름 괄호에서만 얻습니다. 두 소스의 차이가 크면 로더를 소스별로 따로 두고 같은 긴 형식으로 맞춰야 합니다.

In [ ]:
src = ok.groupby("source").agg(
    tables=("key", "count"),
    obs_median=("observations", "median"),
    dims_median=("n_dims", "median"),
    items_median=("n_items", "median"),
    no_unit_tables=("n_units", lambda s: int((s == 0).sum())),
    symbol_tables=("dash", lambda s: int((s > 0).sum())),
    grid_missing_median=("grid_missing_share_max", "median"),
)
display(src)

## 13. 표별 이슈 목록 만들기 (다음 작업용)

**무엇을**: 위에서 본 문제를 표마다 플래그로 모아 `data/eda_full/summary/issues.csv` 로 저장합니다. T1-05(전처리), T1-02(표 선택), 2차(RAG) 작업에서 '어떤 표에 어떤 보정이 필요한지' 찾는 색인으로 씁니다.

**어떻게**: 플래그 정의
- `no_data` / `partial`: 값 없음 / 일부만 수집
- `no_unit`: 값은 있는데 단위가 없음
- `multi_unit_item`: 한 항목에 단위 여러 개
- `symbols`: `-`, `x`, `…`, 잠정치, 기타 문자열 중 하나라도 있음
- `period_gaps`: 주기 안에서 빠진 시점 있음
- `grid_sparse`: 격자의 20% 넘게 비어 있음
- `sum_mismatch`: 더할 수 있는 단위(명·가구·건 등)인데 합계 검증에서 0.1% 이내 일치가 90% 미만인 축이 있음
- `duplicate_keys`: 같은 칸 값이 두 번 이상
- `too_big`: 토큰 추정 128K 초과 (통째로 프롬프트에 넣을 수 없음)

**결과 읽는 법**: 플래그별 표 수와, 플래그가 많은 표 상위 20개가 출력됩니다. 이 숫자가 docs/eda.md 전체 데이터 절의 요약 근거가 됩니다.

In [ ]:
f = pd.DataFrame({"key": ok["key"], "survey": ok["survey"], "title": ok["title"], "source": ok["source"]})
f["no_data"] = ok["data_state"].eq("no_data").values if "data_state" in ok else ok["observations"].eq(0).values
f["partial"] = ok["data_state"].eq("partial").values if "data_state" in ok else False
f["no_unit"] = ((ok["n_units"] == 0) & (ok["observations"] > 0)).values
f["multi_unit_item"] = (ok["items_with_several_units"].fillna(0) > 0).values
f["symbols"] = (ok[["provisional", "dash", "x_secret", "ellipsis", "other_text"]].fillna(0).sum(axis=1) > 0).values
f["period_gaps"] = (ok["period_gaps"].fillna(0) > 0).values
f["grid_sparse"] = (ok["grid_missing_share_max"].fillna(0) > 0.2).values
bad_keys = set(sums[(sums["close_share"] < 0.9) & sums["additive_unit"]]["key"]) if len(sums) else set()
f["sum_mismatch"] = ok["key"].isin(bad_keys).values
f["duplicate_keys"] = (ok["duplicate_keys"].fillna(0) > 0).values
f["too_big"] = (ok["approx_prompt_tokens"].fillna(0) > 128_000).values
flags = [c for c in f.columns if c not in ("key", "survey", "title", "source")]
f["n_flags"] = f[flags].sum(axis=1)
f.to_csv(SUM / "issues.csv", index=False, encoding="utf-8-sig")
print("저장:", SUM / "issues.csv")
print(f[flags].sum().sort_values(ascending=False), "\n")
print("조사별 플래그 수:")
display(f.groupby("survey")[flags].sum())
display(f.sort_values("n_flags", ascending=False).head(20))

## 14. 표 하나 자세히 보기

**무엇을**: 위 목록에서 궁금한 표의 프로파일 json 전체를 봅니다(차원별 코드 예시, 단위, 기호 예시, 시점, 합계 검증 등).

**어떻게**: 아래 `KEY` 를 원하는 표 키(예: `101_DT_1B04005`)로 바꾸고 실행합니다.

**결과 읽는 법**: `read` 는 원본 읽기 정보(인코딩, 파일 수, API 상태), `dims` 는 차원별 구조, `values` 는 값 분포, `sum_checks` 는 합계 검증 상세입니다.

In [ ]:
KEY = "101_DT_1B04005"
p = PROF / f"{KEY}.json"
print(json.dumps(json.loads(p.read_text(encoding="utf-8")), ensure_ascii=False, indent=1)[:8000] if p.exists() else f"없음: {p}")